In [1]:
!apt-get install tesseract-ocr -y
!pip install pytesseract opencv-python pillow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [2]:
import cv2
import pytesseract
from PIL import Image

# Load image
img = cv2.imread('/content/batch1-0001.jpg')

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Apply threshold
thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)[1]

# Save processed image
cv2.imwrite("processed.png", thresh)

# OCR
text = pytesseract.image_to_string(thresh)

print("Extracted Text:")
print(text)

Extracted Text:
Invoice no: 51109338

Date of issue:

Seller:

Andrews, Kirby and Valdez
58861 Gonzalez Prairie
Lake Daniellefurt, IN 57228

Tax Id: 945-82-2137
IBAN: GB75MCRL06841367619257

ITEMS
No. Description Qty
1. CLEARANCE! Fast Dell Desktop 3,00

Computer PC DUAL CORE
WINDOWS 10 4/8/16GB RAM

2. HP T520 Thin Client Computer 5,00
AMD GX-212JC 1.2GHz 4GB RAM
TESTED !!READ BELOW!!

3. gaming pc desktop computer 1,00

4. 12-Core Gaming Computer 3,00
Desktop PC Tower Affordable
GAMING PC 8GB AMD Vega RGB

5. Custom Build Dell Optiplex 9020 5,00
MT i5-4570 3.20GHz Desktop
Computer PC

6. Dell Optiplex 990 MT Computer 4,00
PC Quad Core i7 3.4GHz 16GB
2TB HD Windows 10 Pro

7. Dell Core 2 Duo Desktop 5,00
Computer | Windows XP Pro |
4GB | 500GB
SUMMARY
VAT [%]
10%
Total

04/13/2013

UM

eac

eac

eac

eac

eac

eac

eac

h

 

Client:

Becker Ltd
8012 Stewart Summit Apt. 455
North Douglas, AZ 95355

Tax Id: 942-80-0517

Net price Net worth VAT [%]

209,00 627,00
37,75 188,75
400,00 400

In [3]:
with open("output.txt", "w") as f:
    f.write(text)

print("Text saved to output.txt")

Text saved to output.txt


In [4]:
!unzip dataset.zip

Archive:  dataset.zip
   creating: dataset/
  inflating: dataset/batch1-0001.jpg  
  inflating: dataset/batch1-0003.jpg  
  inflating: dataset/batch1-0004.jpg  
  inflating: dataset/batch1-0005.jpg  
  inflating: dataset/batch1-0006.jpg  
  inflating: dataset/batch1-0007.jpg  
  inflating: dataset/batch1-0008.jpg  
  inflating: dataset/batch1-0009.jpg  
  inflating: dataset/batch1-0010.jpg  
  inflating: dataset/batch1-0011.jpg  
  inflating: dataset/batch1-0012.jpg  
  inflating: dataset/batch1-0013.jpg  
  inflating: dataset/batch1-0014.jpg  
  inflating: dataset/batch1-0015.jpg  
  inflating: dataset/batch1-0016.jpg  
  inflating: dataset/batch1-0017.jpg  
  inflating: dataset/batch1-0018.jpg  
  inflating: dataset/batch1-0019.jpg  
  inflating: dataset/batch1-0020.jpg  
   creating: dataset/processed/


In [5]:
import cv2
import os

input_folder = "/content/dataset"
output_folder = "/content/preprocessed"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    img_path = os.path.join(input_folder, file)
    img = cv2.imread(img_path)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    thresh = cv2.threshold(blur, 150, 255, cv2.THRESH_BINARY)[1]

    out_path = os.path.join(output_folder, file)
    cv2.imwrite(out_path, thresh)

print("Preprocessing completed.")

Preprocessing completed.


In [6]:
import pytesseract
from PIL import Image
import os

ocr_output = {}

for file in os.listdir("/content/preprocessed"):
    img_path = "/content/preprocessed/" + file
    text = pytesseract.image_to_string(Image.open(img_path))
    ocr_output[file] = text

    print("------", file, "------")
    print(text)

------ batch1-0016.jpg ------
Invoice no: 44456646

Date of issue:

Seller:

Austin and Sons
9090 Thomas Route Apt. 293
Johnsonburgh, IA 81799

Tax Id: 916-98-8677
IBAN: GB48PIBU37974446029688

ITEMS
No. Description Qty
1. crewcuts shoes 4,00
2. nickelodeon TMNT Teenage 2,00

Mutant Ninja Turtles DL MR 6in
Action Figure Kids Toy

3. Lands’ End Kids Boys Snow 3,00
Winter Boots Size 5 Blue Warm
Zip Up Easy On Easy Off

4. Under Armour KIDS‘ LOCKDOWN 5,00
4 BASKETBALL Black/Red/White
Youth Sneakers: $z 12k

5. SEVENTEEN Semicolon Special 5,00

Album [Photocard / Digi Pack
Cover / Mini Card)

SUMMARY

VAT [%])
10%

Total

03/21/2015

UM

each

each

each

each

each

Client:

Delacruz, Hinton and Mckinney
3867 Annette Tunnel

North Codybury, FL 61678

Tax Id: 940-84-8016

Net price

12,00

42,98

9,50

19,99

4,99

Net worth
287,36

$ 287,36

Net worth VAT [%])
48,00 10%
85,96 10%
28,50 10%
99,95 10%
24,95 10%

VAT
28,74
$ 28,74

Gross
worth

52,80

94,56

31,35

109,94

27,45

Gross worth

In [8]:
import re

def extract_fields(text):
    invoice_no = re.findall(r'INV[-\s]?\d+', text)
    date = re.findall(r'\d{2}/\d{2}/\d{4}', text)
    amount = re.findall(r'\₹?\$?\d+\.\d{2}', text)

    return {
        "Invoice Number": invoice_no[0] if invoice_no else "Not found",
        "Date": date[0] if date else "Not found",
        "Amount": amount[0] if amount else "Not found"
    }

structured_data = {}

for file, text in ocr_output.items():
    structured_data[file] = extract_fields(text)

structured_data

{'batch1-0016.jpg': {'Invoice Number': 'Not found',
  'Date': '03/21/2015',
  'Amount': 'Not found'},
 'batch1-0001.jpg': {'Invoice Number': 'Not found',
  'Date': '04/13/2013',
  'Amount': '3.20'},
 'batch1-0005.jpg': {'Invoice Number': 'Not found',
  'Date': '10/29/2016',
  'Amount': '3.00'},
 'batch1-0006.jpg': {'Invoice Number': 'Not found',
  'Date': 'Not found',
  'Amount': '3.00'},
 'batch1-0017.jpg': {'Invoice Number': 'Not found',
  'Date': '01/28/2021',
  'Amount': 'Not found'},
 'batch1-0018.jpg': {'Invoice Number': 'Not found',
  'Date': '01/05/2017',
  'Amount': 'Not found'},
 'batch1-0009.jpg': {'Invoice Number': 'Not found',
  'Date': '01/17/2013',
  'Amount': 'Not found'},
 'batch1-0007.jpg': {'Invoice Number': 'Not found',
  'Date': '11/24/2019',
  'Amount': '5.25'},
 'batch1-0020.jpg': {'Invoice Number': 'Not found',
  'Date': '02/18/2014',
  'Amount': 'Not found'},
 'batch1-0012.jpg': {'Invoice Number': 'Not found',
  'Date': 'Not found',
  'Amount': '3.00'},
 'batch

In [10]:
ground_truth = {
    "batch1-0001.jpg": {"Date": "04/13/2013", "Amount": "3.20"},
    "batch1-0005.jpg": {"Date": "10/29/2016", "Amount": "3.00"},
    "batch1-0006.jpg": {"Date": "Not found", "Amount": "3.00"},
    "batch1-0007.jpg": {"Date": "11/24/2019", "Amount": "5.25"},
    "batch1-0008.jpg": {"Date": "01/05/2017", "Amount": "Not found"}
}

In [11]:
correct = 0
total = 0

for file in ground_truth:
    for field in ground_truth[file]:
        total += 1
        if structured_data[file][field] == ground_truth[file][field]:
            correct += 1

accuracy = correct / total
accuracy

0.9

In [12]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 81.4 MB/s eta 0:00:00


In [13]:
%%writefile app.py
import streamlit as st
import pytesseract
import cv2
import re
from PIL import Image
import numpy as np

st.title("OCR Document Processing System")

uploaded_file = st.file_uploader("Upload Invoice Image", type=["jpg","png","jpeg"])

def extract_fields(text):
    invoice_no = re.findall(r'INV[-\s]?\d+', text)
    date = re.findall(r'\d{2}/\d{2}/\d{4}', text)
    amount = re.findall(r'\₹?\$?\d+\.\d{2}', text)

    return {
        "Invoice Number": invoice_no[0] if invoice_no else "Not found",
        "Date": date[0] if date else "Not found",
        "Amount": amount[0] if amount else "Not found"
    }

if uploaded_file:
    image = Image.open(uploaded_file)
    img = np.array(image)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)[1]

    text = pytesseract.image_to_string(thresh)

    st.subheader("Extracted Text")
    st.write(text)

    fields = extract_fields(text)

    st.subheader("Structured Output")
    st.json(fields)

Writing app.py


In [14]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.67.9.246:8501

  Stopping...
^C


In [17]:
structured_data
print(text)

Invoice no: 19471831

Date of issue: 04/09/2014
Seller: Client:
Palmer Ltd Rios, Oneill and Rowe
9790 Bauer Hills Apt. 146 3571 Tina Trafficway
South Patriciaton, SD 32497 Buckleyland, LA 97688
Tax Id: 924-71-1106 Tax Id: 922-72-5979
IBAN: GBO5YUTG50853913677557
ITEMS
No. Description Qty UM Net price Net worth VAT [%] Gross
worth
1. 15"x15" White Decorative 3,00 each 645,77 1 937,31 10% 2:131,04
Coffee Table Top Lapis Lazuli
Marquetery Patio Décor
2. 4'x2' Marble Dining Table Top 5,00 each 1 840,10 9 200,50 10% 10 120,55
Pietra Dura Birds Inlay Art
Furniture Decors B444
3. 60 Inches Marble Dinning Table 5,00 each 5 908,00 29 540,00 10% 32 494,00
Top Hand Inlaid Garden Table
with Gemstones
SUMMARY
VAT [%] Net worth VAT Gross worth
10% 40 677,81 4 067,78 44 745,59

Total $ 40 677,81 $ 4 067,78 $ 44 745,59



In [15]:
 http://34.67.9.246:8501


SyntaxError: invalid syntax (ipython-input-2664498677.py, line 1)